In [1]:
from copy import deepcopy
import math
import random
import numpy as np
import pandas as pd

Simulated annealing - Implementation from course exercise

In [2]:
class Simulated_Annealing:
    def __init__(self,
                 num_iters :int,
                 code:np.ndarray,
                 alpha:float,
                 X_train :pd.DataFrame, 
                 y_train :np.ndarray,
                 calc_accuracy_fn :callable
                ):
        self.num_iters = num_iters
        self.code = code
        self.alpha = alpha
        self.X_train = X_train 
        self.y_train = y_train
        self.calc_acc = calc_accuracy_fn
        self.history = []

    def calc_fitness(self): 
        if not any(self.code):
            return float('-inf')
        acc = self.calc_acc(self.code,self.X_train,self.y_train)
        num_features = sum(self.code)
        return self.alpha * acc + (1 - self.alpha) * (1 - num_features / self.X_train.shape[1])

    
    def run(self):
        best_solution = deepcopy(self.code)
        best_fitness = self.calc_fitness()
        fitness = best_fitness
        for it in range(2, self.num_iters+2):
            for i in range(len(self.code)):
                self.code[i] = not self.code[i]
                new_fitness = self.calc_fitness()
                if new_fitness > fitness:
                    fitness = new_fitness
                    if new_fitness > best_fitness:
                        best_fitness = new_fitness
                        best_solution = deepcopy(self.code)
                    break
                else:
                    p = random.random()
                    q = 1 / (it ** 0.5)
                    if p < q:
                        fitness = new_fitness
                    else:
                        self.code[i] = not self.code[i]
            self.history.append((it,fitness,best_fitness))
        
        number_selected_features = sum(best_solution) 
        best_train_acc = self.calc_acc(best_solution,self.X_train,self.y_train)

        
        return best_solution,best_train_acc,number_selected_features, best_fitness, self.history

Simulated annealing - Extended implementation

In [3]:
class Simulated_Annealing_Extended:
    def __init__(self,
                 num_iters:int,
                 code :list,
                 alpha :float,
                 temperature:float,
                 cooling_schedule:str,
                 cooling_factor :float,
                 X_train :pd.DataFrame,
                 y_train :np.ndarray,
                 calc_accuracy_fn :callable,
                 patience :int
                ):
        self.num_iters = num_iters
        self.code = code
        self.alpha = alpha
        self.temperature = temperature
        self.cooling_schedule = cooling_schedule
        self.cooling_factor = cooling_factor 
        self.X_train = X_train
        self.y_train = y_train
        self.calc_acc = calc_accuracy_fn
        self.no_improve = 0
        self.patience = patience
        
        self.history = []

        
    def calc_fitness(self): 
        if not any(self.code):
            return float('-inf')
        acc = self.calc_acc(self.code,self.X_train,self.y_train)
        num_features = sum(self.code)
        return self.alpha * acc + (1 - self.alpha) * (1 - num_features / self.X_train.shape[1])


    def update_temperature(self, iteration):
        if self.cooling_schedule == 'linear':
            return self.temperature - (self.cooling_factor * iteration)
        elif self.cooling_schedule == 'exponential':
            return self.temperature * (self.cooling_factor ** iteration)
        elif self.cooling_schedule == 'logarithmic':
            return self.temperature / math.log(iteration + 2)
        else:
             raise ValueError("Unsupported cooling schedule")
             
    
    def run(self):
        best_solution = deepcopy(self.code)
        best_fitness = self.calc_fitness()
        fitness = best_fitness
        for it in range(self.num_iters):
            improved = False
            t = self.update_temperature(it)
            #if t < 1e-5 or self.no_improve == self.patience:
            #    break
            indices = list(range(len(self.code))) #da ne ide uvek od pocetka 
            random.shuffle(indices)
            for i in indices:
                self.code[i] = not self.code[i]
                new_fitness = self.calc_fitness()
                if new_fitness > fitness:
                    fitness = new_fitness
                    if new_fitness > best_fitness:
                        best_fitness = new_fitness
                        best_solution = deepcopy(self.code)
                        self.no_improve = 0
                        improved = True
                    break # da li da mu dozvoljim da ide dalje
                else:
                    p = random.uniform(0,1)
                    delta = new_fitness - fitness
                    try:
                        accept_prob = math.exp(delta / t) / 2 #uvek prihvata losije resenje jer je razlika u fitnesu previse mala pa dobijemo e ** 0 otp
                    except OverflowError:
                        accept_prob = 0
                    if p < accept_prob:
                        fitness = new_fitness
                    else:
                        self.code[i] = not self.code[i]
            if improved == False:
                self.no_improve += 1
            self.history.append((it,fitness,best_fitness))
        #selected_percentage = sum(best_solution) / len(best_solution)
        
        number_selected_features = sum(best_solution) 
        best_train_acc = self.calc_acc(best_solution,self.X_train,self.y_train)

        return best_solution, best_train_acc, number_selected_features, best_fitness,self.history